# 08 scenarios rough speed — 不整地・凸凹速度

**対象:** お客様（MPC 設計経験者）との **理論・数式・パラメータ** ディスカッション  
**Part 2/4** — Scenario 06–10

各シナリオは **路面 · 速度 · 勾配 · 実装** を結びつけています。  
理論の前提: [00_theory_grf_mpc_wbc.ipynb](./00_theory_grf_mpc_wbc.ipynb)  
QA 索引: [11_qa_discussion_master.ipynb](./11_qa_discussion_master.ipynb)

```bash
python scripts/scenario_labs.py --list
python scripts/scenario_labs.py --scenario sc06_boxes_s2_gait_fail
```


In [ ]:
import sys
from pathlib import Path

# mpc_dog ルートを sys.path に追加
ROOT = Path.cwd()
for p in [ROOT, *ROOT.parents]:
    if (p / "scripts" / "pympc_lab.py").exists():
        ROOT = p
        break
sys.path.insert(0, str(ROOT / "scripts"))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from pympc_lab import (
    TUNING_GUIDE,
    apply_preset,
    compare_runs,
    load_param_study,
    load_preset_yaml,
    plot_friction_cone,
    run_flat_sim,
    run_speed_terrain_sim,
    run_speed_terrain_sim_resilient,
)

from tuning_labs import (
    TUNING_LABS,
    list_labs,
    run_lab,
    run_lab_pair,
    plot_speed_trial_journey,
    plot_param_study_mu,
    load_cached_lab_results,
)

%matplotlib inline
plt.rcParams["figure.figsize"] = (9, 4)
print(f"repo: {ROOT}")

from scenario_labs import (
    SCENARIO_LABS,
    compare_preset_table,
    run_scenario,
    run_scenario_pair,
    scenario_table,
)


## Scenario 06 — 箱地形 × Session2 gait — 地形持ち込み失敗

| 項目 | 内容 |
|------|------|
| **ID** | `sc06_boxes_s2_gait_fail` |
| **分類** | rough / intermediate |
| **路面** | random_boxes（離散段差） |
| **速度** | — kph |
| **勾配** | flat + 箱 |
| **preset** | `session03_rough_boxes` |

### シナリオ

平坦で調整した step_freq=1.6 を箱地形に持ち込むと数秒で転倒。

### 理論（Layer 2–3）

足場 opt ON でも gait が攻めすぎると $s_i(k)$ 周期内に MPC が安全な $F_i$ を確保できない。

$$\min \sum \|x-x^{ref}\|_Q + \|u\|_R \quad \text{s.t. SRB}, \mu, s_i(k)$$

### パラメータ焦点

| `step_freq` | 1.6 → 1.1 | 箱向け保守化 |

**実装:** `simulation_params.scene=random_boxes`, foothold opt ON

### ノウハウ

地形変更 → freq↓ duty↑ が定石。S2 の勝ち gait を S3 に流用しない。

### 議論用 Q&A

Q: 足場最適化 ON なら gait は攻めて良い？
A: 着地点と GRF は別。gait が速いと支持時間不足。


In [ ]:
from scenario_labs import run_scenario_pair
from pympc_lab import compare_runs

pair = run_scenario_pair("sc06_boxes_s2_gait_fail")
fig = compare_runs(pair)
plt.suptitle("Scenario 06: 箱地形 × Session2 gait — 地形持ち込み失敗", y=1.02)
plt.show()


## Scenario 07 — Perlin × 高μ — 連続起伏での過信

| 項目 | 内容 |
|------|------|
| **ID** | `sc07_perlin_mu_high` |
| **分類** | rough / intermediate |
| **路面** | perlin（連続起伏） |
| **速度** | — kph |
| **勾配** | flat + 起伏 |
| **preset** | `session03_rough_perlin` |

### シナリオ

boxes より滑らかだが、pitch/roll 変動が連続。μ=0.55 は危険。

### 理論（Layer 2–3）

連続地形では $\mathbf{r}_i$ が常に変化 → $\boldsymbol{\tau}=\mathbf{r}_i\times\mathbf{F}_i$ の摂動大。

$$\mathbf{I}\dot{\boldsymbol{\omega}} = \sum \mathbf{r}_i \times \mathbf{F}_i, \quad |F_{t,i}| \le \mu F_{z,i}$$

### パラメータ焦点

| `mu` | 0.55 → 0.42 | perlin 向け |

**実装:** scene=perlin, `use_foothold_optimization=true`

### ノウハウ

perlin は boxes より μ を下げる（0.42 前後）。

### 議論用 Q&A

Q: boxes と perlin で同じ preset で良い？
A: 同系統だが perlin は μ さらに保守。


In [ ]:
from scenario_labs import run_scenario_pair
from pympc_lab import compare_runs

pair = run_scenario_pair("sc07_perlin_mu_high")
fig = compare_runs(pair)
plt.suptitle("Scenario 07: Perlin × 高μ — 連続起伏での過信", y=1.02)
plt.show()


## Scenario 08 — 足場 opt OFF — 地形モデル不一致の切り分け

| 項目 | 内容 |
|------|------|
| **ID** | `sc08_perlin_foothold_off` |
| **分類** | rough / intermediate |
| **路面** | perlin |
| **速度** | — kph |
| **勾配** | flat + 起伏 |
| **preset** | `session03_rough_perlin` |

### シナリオ

不整地で変な足位置 → まず foothold opt OFF で baseline 比較。

### 理論（Layer 2–3）

Layer 2 の foothold 変数が地形推定 $\hat{h}(x,y)$ に依存。不一致で変な $F_i$。

$$u_k^* = \arg\min J(x,u) \quad \text{（foothold ON: } u \text{ に着地も含む）}$$

### パラメータ焦点

| `use_foothold_optimization` | ON vs OFF | 切り分け |

**実装:** `mpc_params.use_foothold_optimization`

### ノウハウ

OFF で安定→地形/推定問題。OFF でも転倒→gait/μ 問題。

### 議論用 Q&A

Q: 本番は常に ON？
A: 推定が信頼できるなら ON。デバッグ・平坦は OFF。


In [ ]:
from scenario_labs import run_scenario_pair
from pympc_lab import compare_runs

pair = run_scenario_pair("sc08_perlin_foothold_off")
fig = compare_runs(pair)
plt.suptitle("Scenario 08: 足場 opt OFF — 地形モデル不一致の切り分け", y=1.02)
plt.show()


## Scenario 09 — 凸凹平坦 3 kph — 速度を下げた no-fall

| 項目 | 内容 |
|------|------|
| **ID** | `sc09_bumpy_3kph` |
| **分類** | speed / intermediate |
| **路面** | bumpy_flat（Perlin heightfield） |
| **速度** | 3.0 kph |
| **勾配** | flat + 凸凹 |
| **preset** | `session04_speed_bumpy_base` |

### シナリオ

5 kph が厳しいとき、3 kph + 長 ramp で no-fall 成功域を確認。

### 理論（Layer 2–3）

指令 $v^{ref}(t)$ ランプ: $v^{ref}(t)=v_{target}\min(t/T_{ramp},1)$。低速度は $F_{ix}$ 要求↓。

$$m a_x \approx \sum F_{ix}, \quad v^{ref} \downarrow \Rightarrow |F_{ix}| \downarrow$$

### パラメータ焦点

| `target_speed_kph` | 3.0 | `speed_ramp_s` | 15 |

**実装:** `run_speed_terrain_sim` — min_distance 10 m で短時間検証

### ノウハウ

速度限界の探索: 3→4→5 kph と段階的に。

### 議論用 Q&A

Q: 3 kph 成功が 5 kph 成功の必要条件？
A: 十分条件ではないが、失敗なら 5 kph は早すぎ。


In [ ]:
from scenario_labs import run_scenario

r = run_scenario("sc09_bumpy_3kph")
res = r.get("result", {})
for k in ("distance_m", "mean_kph", "success", "terminated", "falls", "mean_vx", "max_roll_deg"):
    if k in res:
        print(f"{k}: {res[k]}")


## Scenario 10 — 凸凹 5 kph no-fall — 距離 ~4 m で失敗

| 項目 | 内容 |
|------|------|
| **ID** | `sc10_bumpy_5kph_no_fall_fail` |
| **分類** | speed / advanced |
| **路面** | bumpy_flat |
| **速度** | 5.0 kph |
| **勾配** | flat + 凸凹 |
| **preset** | `session04_speed_bumpy_base` |

### シナリオ

Session 4 の最初の壁。5 kph + 短 ramp → 数 m で転倒。

### 理論（Layer 2–3）

凸凹で $F_{iz}^{min}$ 違反・姿勢摂動。$Q$ で roll/pitch 追従と $R$ で $\|u\|$ のトレードオフ。

$$\min \sum \|x_k-x_k^{ref}\|_Q + \|u_k\|_R \quad \text{s.t. } f_{SRB}, \mu, F_z$$

### パラメータ焦点

| `speed_ramp_s` | 12 → 18 | `step_freq` | 1.35 → 1.20 |

**実装:** `run_speed_terrain_sim` — success は 8 m 到達の緩和版

### ノウハウ

no-fall 不可でも resilient で学習→パラメータ探索に使う。

### 議論用 Q&A

Q: no-fall 20 m は必須？
A: 本ワークショップでは resilient 20 m を実用目標に。


In [ ]:
from scenario_labs import run_scenario_pair
from pympc_lab import compare_runs

pair = run_scenario_pair("sc10_bumpy_5kph_no_fall_fail")
fig = compare_runs(pair)
plt.suptitle("Scenario 10: 凸凹 5 kph no-fall — 距離 ~4 m で失敗", y=1.02)
plt.show()


---

## Part 2 チェックリスト

- [ ] Scenario 06–10 それぞれ **数式 → パラメータ → 結果** を説明できる  
- [ ] fail / OK の差が **摩擦円錐 · gait · 指令 ramp** のどれか特定できる  
- [ ] `configs/pympc_presets/` の YAML と対応づけられる  

**次:** [09_scenarios_slope_ramp.ipynb](./09_scenarios_slope_ramp.ipynb)
